# P1-E02.2 — CAUEEG AD Label Harmonisation and Official No-Overlap Cohort Lock

## Summary

This experiment starts from the raw CAUEEG `annotation.json`, preserves the original diagnostic evidence, and applies deterministic Alzheimer-specific rules. Generic dementia is never relabelled as AD. The strict diagnostic candidates are **459 Normal, 417 MCI, and 230 explicit AD recordings**.

The candidates are then reconciled with the official `dementia-no-overlap.json` partition. The final locked analysis manifest contains **436 Normal, 395 MCI, and 214 AD recordings (1,045 total)** while preserving the official train, validation, and test assignments.

`serial` is treated as a recording identifier. The released files do not contain the serial-to-patient mapping; therefore, this notebook does not claim that serial-disjoint records are independently verified participants. The official no-overlap assignment provides the available patient-overlap protection for held-out evaluation.


## Context & Methods

### Purpose

The raw audit found 1,379 annotation records and 1,379 unique recording serials. Labels are stored as symptom-token lists rather than a single analysis-ready diagnosis. This experiment converts those tokens into reproducible diagnostic categories, excludes non-AD dementias from the AD endpoint, and locks the eligible records to the official patient-overlap-excluded benchmark assignment.

### Key assumptions

1. `annotation.json` is the authoritative raw diagnostic annotation source.
2. `dementia-no-overlap.json` is the authoritative released split for patient-overlap control.
3. `serial` identifies a recording, not an independently verified biological participant.
4. Explicit Alzheimer evidence is required for the AD endpoint.
5. LOAD and EOAD are retained as AD subtypes; vascular, mixed, Parkinson's, and other non-AD dementias are excluded.
6. Age is audited but never used to assign diagnosis.
7. Official train, validation, and test assignments are preserved without reallocation.


### 1. Setup and publication-output contract

In [ ]:
from pathlib import Path
import ast, json, re, sys, platform, hashlib, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

DATA_ROOT=Path(r'D:\EEG_AD_Research\data\caueeg-dataset\caueeg-dataset')
PROJECT_ROOT=Path(r'D:\EEG_AD_Research')
OUTPUT_ROOT=PROJECT_ROOT/'paper1_experiments'/'paper1_outputs'/'p1_e02_label_harmonisation'
FIG_DIR,TABLE_DIR=OUTPUT_ROOT/'figures',OUTPUT_ROOT/'tables'
for directory in (OUTPUT_ROOT,FIG_DIR,TABLE_DIR): directory.mkdir(parents=True,exist_ok=True)

ANNOTATION_FILE=DATA_ROOT/'annotation.json'
OFFICIAL_SPLIT_FILE=DATA_ROOT/'dementia-no-overlap.json'
CLASS_ORDER=['Normal','MCI','AD']
CLASS_COLORS={'Normal':'#0072B2','MCI':'#E69F00','AD':'#D55E00',
              'Dementia-unspecified':'#7A5195','Other/ambiguous':'#8C8C8C'}

mpl.rcParams.update({'font.family':'Arial','font.size':9,'axes.titlesize':10,
    'axes.labelsize':9,'legend.fontsize':8,'savefig.dpi':600,
    'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'ps.fonttype':42})
sns.set_theme(style='ticks',context='paper')

def save_table(frame,name,index=False):
    frame.to_csv(TABLE_DIR/f'{name}.csv',index=index)
    try: frame.to_excel(TABLE_DIR/f'{name}.xlsx',index=index)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')

def save_figure(fig,name,width_mm=183,height_mm=105):
    fig.set_size_inches(width_mm/25.4,height_mm/25.4)
    for extension in ('png','pdf','svg'):
        fig.savefig(FIG_DIR/f'{name}.{extension}',bbox_inches='tight',facecolor='white')

def normalize_serial(value):
    text=str(value).strip()
    if re.fullmatch(r'\d+\.0',text): text=text[:-2]
    digits=''.join(re.findall(r'\d+',text))
    return digits.zfill(5) if digits else text

assert DATA_ROOT.exists(),f'Dataset root does not exist: {DATA_ROOT}'
assert ANNOTATION_FILE.exists(),f'Raw annotation file does not exist: {ANNOTATION_FILE}'
assert OFFICIAL_SPLIT_FILE.exists(),f'Official no-overlap split does not exist: {OFFICIAL_SPLIT_FILE}'
print('Annotation input:',ANNOTATION_FILE)
print('Official split input:',OFFICIAL_SPLIT_FILE)
print('Output:',OUTPUT_ROOT)

## Data

### 2. Read the raw annotation structure

In [ ]:
with ANNOTATION_FILE.open('r',encoding='utf-8-sig') as handle:
    annotation_object=json.load(handle)

if isinstance(annotation_object,dict) and isinstance(annotation_object.get('data'),list):
    raw_records=annotation_object['data']
elif isinstance(annotation_object,list):
    raw_records=annotation_object
else:
    raise ValueError(f'Unsupported annotation.json structure: {type(annotation_object).__name__}; keys={list(annotation_object)[:20] if isinstance(annotation_object,dict) else None}')

raw=pd.json_normalize(raw_records)
required={'serial','age','symptom'}
missing=required-set(raw.columns)
if missing: raise ValueError(f'Missing required raw annotation fields: {sorted(missing)}; available={raw.columns.tolist()}')

raw['serial']=raw['serial'].map(normalize_serial)
raw['age']=pd.to_numeric(raw['age'],errors='coerce')

def canonical_schema_value(value):
    # pandas.nunique cannot hash list/dict/set objects. Convert only for schema counting;
    # the original raw values remain untouched for diagnostic parsing.
    if isinstance(value,(list,tuple,set)):
        return json.dumps(list(value),ensure_ascii=False,sort_keys=True,default=str)
    if isinstance(value,dict):
        return json.dumps(value,ensure_ascii=False,sort_keys=True,default=str)
    return value

def safe_nunique(series):
    normalized=series.map(canonical_schema_value)
    return int(normalized.nunique(dropna=True))

schema=pd.DataFrame({'column':raw.columns,'dtype':[str(raw[c].dtype) for c in raw.columns],
                     'missing':[int(raw[c].isna().sum()) for c in raw.columns],
                     'unique':[safe_nunique(raw[c]) for c in raw.columns]})
save_table(schema,'01_raw_annotation_schema')
display(schema)
print('Rows:',len(raw),'| unique serials:',raw.serial.nunique())

### 3. Parse symptom tokens without changing their meaning

In [ ]:
def parse_tokens(value):
    if value is None or (isinstance(value,float) and np.isnan(value)): return []
    if isinstance(value,(list,tuple,set)): values=list(value)
    else:
        text=str(value).strip()
        try:
            parsed=json.loads(text); values=parsed if isinstance(parsed,list) else [parsed]
        except Exception:
            try:
                parsed=ast.literal_eval(text); values=list(parsed) if isinstance(parsed,(list,tuple,set)) else [parsed]
            except Exception: values=re.split(r'[,;|]',text)
    return sorted({re.sub(r'\s+',' ',str(v).strip().lower()) for v in values if str(v).strip()})

raw['symptom_tokens']=raw['symptom'].map(parse_tokens)
token_counts=Counter(token for tokens in raw.symptom_tokens for token in tokens)
token_table=pd.DataFrame(token_counts.most_common(),columns=['symptom_token','recordings'])
save_table(token_table,'02_symptom_token_vocabulary')
display(token_table.head(50))

### 4. Apply prespecified, Alzheimer-specific label rules

The rules use the token evidence below.

\[
L_i = g(T_i),
\]

where \(T_i\) is the set of raw symptom tokens for recording \(i\), and \(g\) is a deterministic rule. Diagnosis is not inferred from age, EEG, filename, or model predictions.

In [ ]:
AD_EXPLICIT={'ad','alzheimer','alzheimers','alzheimer disease',"alzheimer's disease"}
COMPETING_PREFIXES=('ftd','bvftd','parkinson','nph','tga','sivd','vd','vascular dementia',
                    'non_fluent_aphasia','semantic_aphasia')
MIXED_TOKENS={'ad_vd_mixed','mixed dementia','mixed_dementia'}

def flags_from_tokens(tokens):
    token_set=set(tokens)
    normal=any(t=='normal' or t.endswith('_normal') for t in tokens)
    mci=any(t=='mci' or t.startswith('mci_') or 'mild cognitive impairment' in t for t in tokens)
    # Do not interpret mci_ad as established AD. It remains an MCI token.
    explicit_ad=any(t in AD_EXPLICIT or (t.startswith('ad_') and t not in MIXED_TOKENS) for t in tokens)
    dementia=any(t=='dementia' or t.startswith('dementia_') for t in tokens)
    mixed=any(t in MIXED_TOKENS or 'mixed' in t for t in tokens)
    competing=any(any(t==p or t.startswith(p+'_') for p in COMPETING_PREFIXES) for t in tokens)
    return normal,mci,explicit_ad,dementia,mixed,competing

flag_columns=['flag_normal','flag_mci','flag_explicit_ad','flag_dementia','flag_mixed','flag_competing']
raw[flag_columns]=pd.DataFrame(raw.symptom_tokens.map(flags_from_tokens).tolist(),index=raw.index)

def decide_label(row):
    positive=sum([row.flag_normal,row.flag_mci,row.flag_explicit_ad])
    if positive>1: return 'Other/ambiguous','conflicting_primary_tokens'
    if row.flag_mixed: return 'Other/ambiguous','mixed_diagnosis'
    if row.flag_competing: return 'Other/ambiguous','competing_neurological_diagnosis'
    if row.flag_explicit_ad: return 'AD','eligible_explicit_ad'
    if row.flag_mci and not row.flag_dementia: return 'MCI','eligible_explicit_mci'
    if row.flag_normal and not row.flag_dementia: return 'Normal','eligible_explicit_normal'
    if row.flag_dementia: return 'Dementia-unspecified','dementia_without_explicit_ad'
    return 'Other/ambiguous','unsupported_or_unclassified_tokens'

decisions=raw.apply(decide_label,axis=1,result_type='expand')
decisions.columns=['harmonised_class','label_reason']
audit=pd.concat([raw,decisions],axis=1)
audit['strict_three_class_eligible']=audit.harmonised_class.isin(CLASS_ORDER)
audit['primary_label']=audit.harmonised_class.where(audit.strict_three_class_eligible,pd.NA)
audit['age_plausible']=audit.age.between(18,110,inclusive='both')
audit['tokens_text']=audit.symptom_tokens.map('|'.join)

label_decision=audit.groupby(['harmonised_class','label_reason'],dropna=False).size().rename('recordings').reset_index()
save_table(label_decision,'03_label_decision_counts')
save_table(audit.drop(columns=['symptom_tokens']),'04_recording_label_audit')
display(label_decision.sort_values(['harmonised_class','recordings'],ascending=[True,False]))

## Results

### 5. Reconcile label evidence and conflicts

In [ ]:
cooccurrence=(audit[flag_columns].astype(int).T @ audit[flag_columns].astype(int))
cooccurrence.index=cooccurrence.index.str.replace('flag_','',regex=False)
cooccurrence.columns=cooccurrence.columns.str.replace('flag_','',regex=False)
save_table(cooccurrence.reset_index(names='evidence'),'05_label_evidence_cooccurrence')

fig,ax=plt.subplots()
sns.heatmap(cooccurrence,annot=True,fmt='d',cmap='Blues',cbar_kws={'label':'Recordings'},ax=ax)
ax.set(title='Co-occurrence of raw diagnostic evidence',xlabel='Evidence flag',ylabel='Evidence flag')
save_figure(fig,'figure_1_diagnostic_evidence_cooccurrence',150,120)
plt.show()

conflicts=audit[(audit[flag_columns].sum(axis=1)>1)|audit.harmonised_class.eq('Other/ambiguous')]
save_table(conflicts.drop(columns=['symptom_tokens']),'06_conflicting_or_ambiguous_labels')
print('Conflict/ambiguity review rows:',len(conflicts))
display(conflicts[['serial','age','tokens_text','harmonised_class','label_reason']].head(30))

### 6. Describe the strict candidate classes and age distributions

In [ ]:
strict=audit[audit.strict_three_class_eligible].copy()
counts=strict.primary_label.value_counts().reindex(CLASS_ORDER,fill_value=0)
age_summary=(strict.groupby('primary_label').age.agg(['count','mean','std','median','min','max'])
             .reindex(CLASS_ORDER).reset_index())
save_table(age_summary,'07_age_summary_strict_candidate_classes')

fig,axes=plt.subplots(1,2)
bars=axes[0].bar(counts.index,counts.values,color=[CLASS_COLORS[c] for c in counts.index])
axes[0].bar_label(bars,padding=2)
axes[0].set(xlabel='Diagnostic class',ylabel='Recordings',title='Strict label-harmonised candidates')
for label in CLASS_ORDER:
    values=strict.loc[strict.primary_label.eq(label),'age'].dropna()
    if len(values): sns.kdeplot(x=values,label=label,color=CLASS_COLORS[label],ax=axes[1],warn_singular=False)
axes[1].set(xlabel='Age (years)',ylabel='Density',title='Age distribution before age control')
axes[1].legend(title='Class')
save_figure(fig,'figure_2_strict_class_counts_and_age',183,82)
plt.show()
display(age_summary)

### 7. Quantify age imbalance without using age to alter labels

In [ ]:
def standardized_mean_difference(x,y):
    x=pd.Series(x).dropna().astype(float); y=pd.Series(y).dropna().astype(float)
    pooled=np.sqrt((x.var(ddof=1)+y.var(ddof=1))/2)
    return np.nan if pooled==0 or not np.isfinite(pooled) else (x.mean()-y.mean())/pooled

age_smd=[]
for a,b in [('AD','Normal'),('AD','MCI'),('MCI','Normal')]:
    age_smd.append({'comparison':f'{a} vs {b}',
                    'age_SMD':standardized_mean_difference(strict.loc[strict.primary_label.eq(a),'age'],
                                                           strict.loc[strict.primary_label.eq(b),'age'])})
age_smd=pd.DataFrame(age_smd)
save_table(age_smd,'08_pairwise_age_standardized_mean_differences')
display(age_smd)
print('Interpretation: |SMD| ≥ 0.10 indicates imbalance requiring fold-safe age control; it is not a reason to relabel participants.')

### 8. Create the complete label-harmonisation audit and initial quality gates

The full 1,379-record manifest remains an audit artifact. It is not the final analysis cohort.


In [ ]:
manifest_columns=['serial','age','symptom','tokens_text']+flag_columns+[
    'harmonised_class','label_reason','strict_three_class_eligible','primary_label','age_plausible']
manifest=audit[manifest_columns].copy()
manifest=manifest.sort_values('serial').reset_index(drop=True)
save_table(manifest,'09_complete_label_harmonisation_manifest')

checks=pd.DataFrame([
    {'check':'raw annotation rows loaded','pass':len(raw)==1379,'detail':len(raw)},
    {'check':'recording serials nonblank','pass':raw.serial.ne('').all(),'detail':int(raw.serial.eq('').sum())},
    {'check':'recording serials unique in annotation','pass':not raw.serial.duplicated().any(),'detail':int(raw.serial.duplicated().sum())},
    {'check':'every row receives an audited category','pass':audit.harmonised_class.notna().all(),'detail':audit.harmonised_class.value_counts().to_dict()},
    {'check':'generic dementia excluded from AD endpoint','pass':not audit.loc[~audit.flag_explicit_ad,'primary_label'].eq('AD').any(),'detail':int((~audit.flag_explicit_ad & audit.primary_label.eq('AD')).sum())},
    {'check':'strict labels contain only Normal/MCI/AD','pass':set(strict.primary_label.dropna()).issubset(set(CLASS_ORDER)),'detail':strict.primary_label.value_counts().to_dict()},
    {'check':'strict candidate counts reproduce audit','pass':strict.primary_label.value_counts().reindex(CLASS_ORDER,fill_value=0).to_dict()=={'Normal':459,'MCI':417,'AD':230},'detail':strict.primary_label.value_counts().to_dict()},
    {'check':'ages are plausible or explicitly missing','pass':audit.loc[audit.age.notna(),'age_plausible'].all(),'detail':int((audit.age.notna()&~audit.age_plausible).sum())},
])
save_table(checks,'10_initial_label_harmonisation_checks')
display(checks)
initial_status='INITIAL LABEL HARMONISATION PASSED' if checks['pass'].all() else 'STOP — REVIEW FAILED INITIAL LABEL CHECKS'
print(initial_status)
if not checks['pass'].all():
    raise RuntimeError(initial_status)


### 9. Reconcile strict diagnostic candidates with the official no-overlap split

The official split is joined by recording serial. Its assignments are preserved exactly. This step supplies the released patient-overlap protection but does not reconstruct the unavailable patient mapping.


In [ ]:
with OFFICIAL_SPLIT_FILE.open('r',encoding='utf-8-sig') as handle:
    official_object=json.load(handle)

split_key_map={
    'train_split':'train',
    'validation_split':'validation',
    'test_split':'test',
}
official_parts=[]
for source_key,split_name in split_key_map.items():
    if source_key not in official_object:
        raise KeyError(f'Missing official split key: {source_key}')
    part=pd.json_normalize(official_object[source_key])
    required_split_fields={'serial','class_name','class_label'}
    missing_split_fields=required_split_fields-set(part.columns)
    if missing_split_fields:
        raise ValueError(f'{source_key} is missing fields: {sorted(missing_split_fields)}')
    part['serial']=part.serial.map(normalize_serial)
    part['official_split']=split_name
    official_parts.append(part[['serial','official_split','class_name','class_label']])

official_manifest=pd.concat(official_parts,ignore_index=True)
if official_manifest.serial.duplicated().any():
    duplicates=official_manifest.loc[official_manifest.serial.duplicated(False),['serial','official_split']]
    raise ValueError(f'Official partitions are not serial-disjoint:\n{duplicates.head(20)}')

expected_split_sizes={'train':950,'validation':82,'test':90}
observed_split_sizes=official_manifest.official_split.value_counts().to_dict()
if observed_split_sizes!=expected_split_sizes:
    raise AssertionError(f'Unexpected official split sizes: {observed_split_sizes}')

locked=(official_manifest.merge(
    manifest,
    on='serial',
    how='left',
    validate='one_to_one',
    indicator=True,
))
if not locked['_merge'].eq('both').all():
    raise ValueError('Some official split serials did not match annotation.json.')
locked=locked.drop(columns='_merge')

# Verify that the broad official class is compatible with the raw parent evidence.
broad_consistency=(
    (locked.class_name.eq('Normal') & locked.flag_normal) |
    (locked.class_name.eq('MCI') & locked.flag_mci) |
    (locked.class_name.eq('Dementia') & locked.flag_dementia)
)
if not broad_consistency.all():
    raise ValueError('Official broad labels disagree with raw annotation evidence.')

official_label_audit=locked.copy()
locked=locked[locked.strict_three_class_eligible].copy()
locked['experimental_label']=locked.primary_label
locked['experimental_label_code']=locked.experimental_label.map({'Normal':0,'MCI':1,'AD':2}).astype(int)
locked['analysis_unit']='recording'
locked['identity_group']=locked.serial
locked['participant_identity_verified']=False
locked['split_source']='CAUEEG dementia-no-overlap.json'

locked_columns=[
    'serial','age','official_split','experimental_label','experimental_label_code',
    'analysis_unit','identity_group','participant_identity_verified','split_source',
    'symptom','tokens_text','flag_normal','flag_mci','flag_explicit_ad',
    'flag_dementia','flag_mixed','flag_competing','label_reason'
]
locked=locked[locked_columns].sort_values(['official_split','experimental_label','serial']).reset_index(drop=True)

official_exclusions=official_label_audit[~official_label_audit.strict_three_class_eligible].copy()
save_table(official_label_audit.drop(columns=[]),'11_official_no_overlap_label_audit')
save_table(official_exclusions,'12_official_no_overlap_diagnostic_exclusions')
save_table(locked,'13_LOCKED_NORMAL_MCI_AD_OFFICIAL_NO_OVERLAP_COHORT')

locked_counts=(locked.groupby(['official_split','experimental_label']).size()
               .unstack(fill_value=0).reindex(index=['train','validation','test'],columns=CLASS_ORDER,fill_value=0))
locked_counts['Total']=locked_counts.sum(axis=1)
locked_counts.loc['Total']=locked_counts.sum(axis=0)
save_table(locked_counts.reset_index(),'14_locked_cohort_counts')
display(locked_counts)


### 10. Apply final cohort-lock quality gates

These gates stop execution if the class definitions, official assignments, or expected counts change.


In [ ]:
expected_locked_counts=pd.DataFrame(
    {'Normal':[367,34,35],'MCI':[334,28,33],'AD':[189,10,15]},
    index=['train','validation','test']
)
observed_locked_counts=locked_counts.loc[['train','validation','test'],CLASS_ORDER].astype(int)

final_checks=pd.DataFrame([
    {'check':'official no-overlap records loaded','pass':len(official_manifest)==1122,'detail':len(official_manifest)},
    {'check':'official split serials unique','pass':not official_manifest.serial.duplicated().any(),'detail':int(official_manifest.serial.duplicated().sum())},
    {'check':'official assignments preserved','pass':observed_split_sizes==expected_split_sizes,'detail':observed_split_sizes},
    {'check':'locked cohort contains 1,045 recordings','pass':len(locked)==1045,'detail':len(locked)},
    {'check':'locked serials unique','pass':locked.serial.nunique()==len(locked),'detail':int(locked.serial.nunique())},
    {'check':'locked counts reproduce audited values','pass':observed_locked_counts.equals(expected_locked_counts),'detail':observed_locked_counts.to_dict()},
    {'check':'AD requires explicit AD evidence','pass':locked.loc[locked.experimental_label.eq('AD'),'flag_explicit_ad'].all(),'detail':int((locked.experimental_label.eq('AD')&~locked.flag_explicit_ad).sum())},
    {'check':'non-AD dementia excluded from endpoint','pass':not locked.flag_competing.any() and not locked.flag_mixed.any(),'detail':{'competing':int(locked.flag_competing.sum()),'mixed':int(locked.flag_mixed.sum())}},
    {'check':'test split retained but not reallocated','pass':set(locked.official_split)=={'train','validation','test'},'detail':locked.official_split.value_counts().to_dict()},
])
save_table(final_checks,'15_final_cohort_lock_checks')
display(final_checks)

status='CORRECTION 3 PASSED — NORMAL/MCI/AD OFFICIAL NO-OVERLAP COHORT LOCKED' if final_checks['pass'].all() else 'STOP — FINAL COHORT LOCK FAILED'
print(status)
if not final_checks['pass'].all():
    raise RuntimeError(status)


### 11. Save the cohort flow, figure, and reproducibility record


In [ ]:
flow=pd.DataFrame([
    {'stage':'Raw annotation records','n':len(audit)},
    {'stage':'Strict Normal/MCI/AD diagnostic candidates','n':int(audit.strict_three_class_eligible.sum())},
    {'stage':'Official no-overlap benchmark records','n':len(official_manifest)},
    {'stage':'Locked Normal/MCI/AD records','n':len(locked)},
])
save_table(flow,'16_label_and_official_split_flow_counts')

fig,ax=plt.subplots()
colors=['#4C78A8','#59A14F','#7A5195','#D55E00']
bars=ax.barh(flow.stage[::-1],flow.n[::-1],color=colors[::-1])
ax.bar_label(bars,padding=3)
ax.set(xlabel='Recordings',ylabel='',title='CAUEEG diagnosis and official-split cohort flow')
save_figure(fig,'figure_3_label_harmonisation_and_cohort_lock_flow',183,95)
plt.show()

input_hash=hashlib.sha256(ANNOTATION_FILE.read_bytes()).hexdigest()
split_hash=hashlib.sha256(OFFICIAL_SPLIT_FILE.read_bytes()).hexdigest()
locked_csv=TABLE_DIR/'13_LOCKED_NORMAL_MCI_AD_OFFICIAL_NO_OVERLAP_COHORT.csv'
locked_hash=hashlib.sha256(locked_csv.read_bytes()).hexdigest()
run_record={
    'experiment':'P1-E02.2 CAUEEG AD label harmonisation and official no-overlap cohort lock',
    'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
    'inputs':{
        'annotation':str(ANNOTATION_FILE),'annotation_sha256':input_hash,
        'official_no_overlap_split':str(OFFICIAL_SPLIT_FILE),'official_split_sha256':split_hash,
    },
    'raw_annotation_records':len(audit),
    'strict_diagnostic_candidates':int(audit.strict_three_class_eligible.sum()),
    'official_no_overlap_records':len(official_manifest),
    'locked_recordings':len(locked),
    'locked_class_counts':{str(k):int(v) for k,v in locked.experimental_label.value_counts().items()},
    'locked_split_counts':{str(k):int(v) for k,v in locked.official_split.value_counts().items()},
    'identity_interpretation':{
        'serial':'recording identifier',
        'serial_to_patient_mapping_available':False,
        'internal_claim':'recording-disjoint',
        'held_out_protection':'official CAUEEG no-overlap assignment',
    },
    'locked_manifest':str(locked_csv),'locked_manifest_sha256':locked_hash,
    'status':status,'python':sys.version,'platform':platform.platform(),
}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))


## Takeaways

1. Explicit AD evidence produces 230 AD diagnostic candidates; generic and competing dementias are not converted to AD.
2. The official no-overlap split reduces the final analysis manifest to 1,045 recordings: 436 Normal, 395 MCI, and 214 AD.
3. Official train, validation, and test assignments are preserved exactly.
4. `serial` remains a recording identifier. The hidden serial-to-patient mapping cannot be reconstructed from the released files.
5. The locked file `13_LOCKED_NORMAL_MCI_AD_OFFICIAL_NO_OVERLAP_COHORT.csv` is the only cohort input permitted for downstream linkage, preprocessing selection, representation extraction, and benchmarking.

